In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

from src.models import SIRM, SIRT, SIRV
from helps import *




from src.utils.batch_sweep import sweep_two_parameters

# Import the matrix creation function
from src.utils.Contact_Matrix import create_contact_matrix




NB = 30
NP = 30

# Things for Fig. 1

- 3 symmetrical beta distributions with the purple-to-green
- 3 asymmetrical ones
- 3 contact matrices
- colorbar blues,
- colorbar green-to-purple

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Your custom colorscheme
colors = ['#8e0152', '#c51b7d', '#de77ae', '#f1b6da', '#fde0ef', 
          '#f7f7f7', '#e6f5d0', '#b8e186', '#7fbc41', '#4d9221', '#276419']

# Create a custom colormap using your colors
cmap_name = 'custom_diverging'
custom_cmap = LinearSegmentedColormap.from_list(cmap_name, colors)

# Create a figure and axis
fig, ax = plt.subplots(figsize=(10, 1))

# Create data for the colorbar
gradient = np.linspace(0, 1, 256)
gradient = np.vstack((gradient, gradient))

# Display the colorbar
ax.imshow(gradient, aspect='auto', cmap=custom_cmap)

# Remove ticks from the plot
ax.set_yticks([])
ax.set_xticks(np.linspace(0, 256, 3))
ax.set_xticklabels([])


# Optional: Add a border to the colorbar
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color('black')
    spine.set_linewidth(0.5)



fig.savefig('figures/Fig_1/custom_colorbar.pdf', dpi=300, bbox_inches='tight')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import beta
from matplotlib.collections import LineCollection
def plot_beta_with_gradient(alpha, beta_param, Nbins=100):
    fig, ax = plt.subplots(figsize=(Lx/3, Ly/5))
    
    x = np.linspace(1/Nbins/2, 1-1/Nbins/2, Nbins)
    y = beta.pdf(x, alpha, beta_param)

    points = np.array([x, y]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    
    lc = LineCollection(segments, cmap=custom_cmap, norm=plt.Normalize(0, 1))
    lc.set_array(x[:-1])
    lc.set_linewidth(3)
    
    line = ax.add_collection(lc)
    ax.plot(x, y, '--', color='black', linewidth=1)
       
    ax.set_xlim(-0.01, 1.01)
    ax.set_ylim(-0.05, 5.05)
    # remove top and right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    

    ax.set_xticks([])
    ax.set_xticklabels([])
    ax.set_yticks([])
    ax.set_yticklabels([])
    fig.patch.set_visible(False)
    plt.show()



    return fig, ax

In [ ]:
fig, ax = plot_beta_with_gradient(5, 5, Nbins=1000)  # Symmetric
fig.savefig('figures/Fig_1/beta_sym_1.pdf', dpi=300, bbox_inches='tight')
fig, ax = plot_beta_with_gradient(1, 1, Nbins=1000)  # Symmetric
fig.savefig('figures/Fig_1/beta_sym_2.pdf', dpi=300, bbox_inches='tight')
fig, ax = plot_beta_with_gradient(0.1, 0.1, Nbins=1000)  # Symmetric
fig.savefig('figures/Fig_1/beta_sym_3.pdf', dpi=300, bbox_inches='tight')

fig, ax = plot_beta_with_gradient(5, 0.1, Nbins=1000)  # Symmetric
fig.savefig('figures/Fig_1/beta_asym_1.pdf', dpi=300, bbox_inches='tight')
fig, ax = plot_beta_with_gradient(1, 1, Nbins=1000)  # Symmetric
fig.savefig('figures/Fig_1/beta_asym_2.pdf', dpi=300, bbox_inches='tight')
fig, ax = plot_beta_with_gradient(0.1, 5, Nbins=1000)  # Symmetric
fig.savefig('figures/Fig_1/beta_asym_3.pdf', dpi=300, bbox_inches='tight')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax.numpy as jnp
from typing import List, Optional, Tuple

def plot_contact_matrices(h_values: List[float], 
                         n_groups: int = 5,
                         figsize: Tuple[int, int] = (15, 3),
                         cmap: str = "Blues",
                         save_path: Optional[str] = None,
                         text_flag = True, vmin = 0, vmax = 3) -> plt.Figure:
    """
    Plot contact matrices for different values of homophilic tendency (h).
    
    Args:
        h_values: List of homophilic tendency values to visualize
        n_groups: Number of population groups (matrix size will be n_groups x n_groups)
        figsize: Figure size (width, height)
        cmap: Colormap to use
        save_path: Path to save the figure (if None, figure is not saved)
        
    Returns:
        Matplotlib figure object
    """
    
    # Create a figure with subplots for each h value
    n_plots = len(h_values)
    fig, axes = plt.subplots(1, n_plots, figsize=figsize)
    
    # If only one h value, make axes iterable
    if n_plots == 1:
        axes = [axes]
    
    # Create an equal population distribution
    pop = jnp.ones(n_groups)*0.2
    
    # For each h value, create and plot the contact matrix
    for i, h in enumerate(axes):
        # Create contact matrix
        h_val = h_values[i]
        C = create_contact_matrix(n_groups, h_val, pop)
        C = np.flipud(C)
        # Plot as heatmap
        im = axes[i].imshow(C, cmap=cmap, vmin=vmin, vmax=vmax, extent=[-0.5, n_groups-0.5, -0.5, n_groups-0.5])
        
        if text_flag:
            # Add text annotations to each cell
            for row in range(n_groups):
                for col in range(n_groups):
                    value = np.round(C[row, col],1)
                    # Add text with value to each cell
                    axes[i].text(col, row, f"{value:.1f}", ha="center", va="center", 
                            color="black" if value < 1.5 else "white", fontsize=8)
        
        axes[i].set_xticklabels([])
        axes[i].set_yticklabels([])        
        axes[i].grid(False)
        
        # Add h value as title
        #axes[i].set_title(f"h = {h_val}")
    
    fig.patch.set_visible(False)
    
    # add a grid to separate pixels in the heatmap
    for ax in axes:
        ax.set_xticks(np.arange(0.5, n_groups, 1), minor=True)
        ax.set_yticks(np.arange(0.5, n_groups, 1), minor=True)
        ax.grid(which='minor', color='black', linestyle='-', linewidth=0.5)
        ax.tick_params(which="minor", bottom=False, left=False)


    
    # Save if requested
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    return fig

In [ ]:

from src.utils.Contact_Matrix import create_contact_matrix

h_values = [0]  # Different homophilic tendency values
fig = plot_contact_matrices(h_values, n_groups=5, save_path='figures/Fig_1/contact_matrices0.pdf', text_flag=False)
plt.show()

h_values = [2]  # Different homophilic tendency values
fig = plot_contact_matrices(h_values, n_groups=5, save_path='figures/Fig_1/contact_matrices2.pdf', text_flag=False)
plt.show()

h_values = [4]  # Different homophilic tendency values
fig = plot_contact_matrices(h_values, n_groups=5, save_path='figures/Fig_1/contact_matrices4.pdf', text_flag=False)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

def make_colorbar(cmap, vmin=0, vmax=1):
    """Generate only a colorbar with specified colormap
    
    Args:
        cmap: colormap object or string name
    """
    
    fig = plt.figure(figsize=(1 / 25.4, 100 / 25.4))
    ax = fig.add_axes([0.05, 0.4, 0.9, 0.2])
    
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    
    if isinstance(cmap, str):
        cmap = mpl.colormaps[cmap]
    
    cb = mpl.colorbar.ColorbarBase(ax, cmap=cmap, norm=norm, orientation='vertical')
    
    ax.set_yticks([0,0.5,1])
    ax.set_yticklabels([])

    ax.tick_params(width=0.5)
    fig.patch.set_visible(False)
    # reduce thickness of colorbar border
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)
    return fig

fig_cb = make_colorbar("Blues", vmin=0, vmax=1)
fig_cb.savefig('figures/Fig_1/colorbar_contact_matrix.pdf', dpi=300, bbox_inches='tight')

# Thinkgs for Fig. 2

## General parameters first

In [ ]:
temp = read_json("./parameters_test.json")
mus, taus, xis, PARAMS = temp["mus"], temp["taus"], temp["xis"], temp["PARAMS"]
rect_coords_M = [mus["pol"][0], mus["h"][0], mus["pol"][2]-mus["pol"][0], mus["h"][2]-mus["h"][0]]
rect_coords_T = [taus["pol"][0], taus["h"][0], taus["pol"][2]-taus["pol"][0], taus["h"][2]-taus["h"][0]]
rect_coords_V = [xis["pol"][0], xis["h"][0], xis["pol"][2]-xis["pol"][0], xis["h"][2]-xis["h"][0]]

PARAMS["fixed_mean"] = 0.5
# visualization parameters

colors_X = ['#66c2a4', '#238b45','#00441b']  # fixed polarization
colors_Y = ['#67001f', '#e7298a', '#df65b0'] # fixed homophily
my_map = discretize_cmaps("hot_r",51)
my_map.set_bad(color='gray')
cmaps = [my_map]
contour_values = [[0.2, 0.4, 0.5, 0.6, 0.8]]
contour_colors = [['#000','#000','#000']]
final_params={
        'Lx': Lx/1.25,  # Figure width in inches
        'Ly': Ly/1.25,  # Figure height in inches
        'xticks': [0, 0.5, 1.0],
        'yticks': [0, 3, 6],
        'xlim': [0, 1],
        'ylim': [0, 6]
    }

## 9 plots

In [ ]:
homophilic_tendency = {"m": 0, "M": 6, "n": NB}
pol_range = {"m": 0, "M": 1, "n": NP}

PARAMS["fixed_mean"] = 0.5
betas = [0.15, 0.25, 0.4]
print("### SIR-M ###")
PS = 5
res_list_M_2 = []
for b in betas:
    PARAMS["beta_M"] = b
    M = sweep_two_parameters(
        model_module=SIRM,
        param1_name="beta_params",           # parameter 1 name
        param1_range=pol_range,    # parameter 1 range
        param2_name="homophilic_tendency",      # parameter 2 name
        param2_range=homophilic_tendency,         # parameter 2 range
        custom_base_params=PARAMS,
        simulated_days=1000,
        population_size=PS,
        batch_size=2000
    )
    res_list_M_2.append(M)
    print(f"Completed beta_M = {b}")

print("### SIR-T ###")
res_list_T_2 = []

for b in betas:
    PARAMS["beta_M"] = b
    M = sweep_two_parameters(
        model_module=SIRT,
        param1_name="beta_params",           # parameter 1 name
        param1_range=pol_range,    # parameter 1 range
        param2_name="homophilic_tendency",      # parameter 2 name
        param2_range=homophilic_tendency,         # parameter 2 range
        custom_base_params=PARAMS,
        simulated_days=1000,
        population_size=PS,
        batch_size=2000
    )
    res_list_T_2.append(M)
    print(f"Completed beta_M = {b}")

print("### SIR-V ###")
res_list_V_2 = []
for b in betas:
    PARAMS["beta_M"] = b
    M = sweep_two_parameters(
        model_module=SIRV,
        param1_name="beta_params",           # parameter 1 name
        param1_range=pol_range,    # parameter 1 range
        param2_name="homophilic_tendency",      # parameter 2 name
        param2_range=homophilic_tendency,         # parameter 2 range
        custom_base_params=PARAMS,
        simulated_days=1000,
        population_size=PS,
        batch_size=2000
    )
    res_list_V_2.append(M)
    print(f"Completed beta_M = {b}")


In [ ]:
path_1_M_L = "figures/Fig_2/masks_L_08.pdf"
path_1_M_M = "figures/Fig_2/masks_M_08.pdf"
path_2_M_H = "figures/Fig_2/masks_H_08.pdf"

contour_values = [[0.2, 0.4, 0.5, 0.6, 0.8]]
fig_R_M_1_L = plot_multiple_metrics(
    res_list_M_2[0], 
    metrics=["infections"],
    cmaps=cmaps, 
    contour_values=contour_values,
    contour_colors=contour_colors,
    final_params=final_params,
    save_path=path_1_M_L
)
CV = [[0.4, 0.5, 0.6]]
fig_R_M_1_H = plot_multiple_metrics(
    res_list_M_2[1], 
    metrics=["infections"],
    cmaps=cmaps, 
    contour_values=CV,
    contour_colors=contour_colors,
    final_params=final_params,
    save_path=path_1_M_M
)
fig_R_M_2 = plot_multiple_metrics(
    res_list_M_2[2], 
    metrics=["infections"],
    cmaps=cmaps, 
    contour_values=contour_values,
    contour_colors=contour_colors,
    final_params=final_params,
    save_path=path_2_M_H
)

In [ ]:
path_1_T_L = "figures/Fig_2/tests_L.pdf"
path_1_T_M = "figures/Fig_2/tests_M.pdf"
path_1_T_H = "figures/Fig_2/tests_H.pdf"


fig_R_M_1_L = plot_multiple_metrics(
    res_list_T_2[0], 
    metrics=["infections"],
    cmaps=cmaps, 
    contour_values=contour_values,
    contour_colors=contour_colors,
    final_params=final_params,
    save_path=path_1_T_L
)
fig_R_M_1_H = plot_multiple_metrics(
    res_list_T_2[1], 
    metrics=["infections"],
    cmaps=cmaps, 
    contour_values=contour_values,
    contour_colors=contour_colors,
    final_params=final_params,
    save_path=path_1_T_M
)
fig_R_M_2 = plot_multiple_metrics(
    res_list_T_2[2], 
    metrics=["infections"],
    cmaps=cmaps, 
    contour_values=contour_values,
    contour_colors=contour_colors,
    final_params=final_params,
    save_path=path_1_T_H
)

In [ ]:
path_1_V_L = "figures/Fig_2/vaccines_L.pdf"
path_1_V_M = "figures/Fig_2/vaccines_M.pdf"
path_1_V_H = "figures/Fig_2/vaccines_H.pdf"


fig_R_M_1_L = plot_multiple_metrics(
    res_list_V_2[0], 
    metrics=["infections"],
    cmaps=cmaps, 
    contour_values=contour_values,
    contour_colors=contour_colors,
    final_params=final_params,
    save_path=path_1_V_L
)
fig_R_M_1_H = plot_multiple_metrics(
    res_list_V_2[1], 
    metrics=["infections"],
    cmaps=cmaps, 
    contour_values=contour_values,
    contour_colors=contour_colors,
    final_params=final_params,
    save_path=path_1_V_M
)
fig_R_M_2 = plot_multiple_metrics(
    res_list_V_2[2], 
    metrics=["infections"],
    cmaps=cmaps, 
    contour_values=contour_values,
    contour_colors=contour_colors,
    final_params=final_params,
    save_path=path_1_V_H
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

def make_colorbar(cmap, vmin=0, vmax=1):
    """Generate only a colorbar with specified colormap
    
    Args:
        cmap: colormap object or string name
    """
    
    fig = plt.figure(figsize=(34 / 25.4, 10 / 25.4))
    ax = fig.add_axes([0.05, 0.4, 0.9, 0.2])
    
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    
    if isinstance(cmap, str):
        cmap = mpl.colormaps[cmap]
    
    cb = mpl.colorbar.ColorbarBase(ax, cmap=cmap, norm=norm, orientation='horizontal')
    
    ax.set_xticks([0,0.5,1])
    ax.tick_params(width=0.5)
    ax.set_xticklabels([])
    fig.patch.set_visible(False)
    # reduce thickness of colorbar border
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)
    return fig

fig_cb = make_colorbar(my_map, vmin=0, vmax=1)
fig_cb.savefig('figures/Fig_2/colorbar_hot_r.pdf', dpi=300, bbox_inches='tight')

# Things for Fig. 3:
- Real contact matrices
- Real Behavior distribution
- Fraction of infected with real average
- Prediction error
- colorbar heat
- colorbar blues

In [ ]:
from bootstrap import *
df = pd.read_csv("data_homophily.csv")
df = df.dropna()

mask_distribution = extract_behavior_distribution_vectorized(df, "masks")
test_distribution = extract_behavior_distribution_vectorized(df, "testing")
vaccine_distribution = extract_behavior_distribution_vectorized(df, "vacc")

mask_matrix = generate_contact_matrix(df, "masks")
test_matrix = generate_contact_matrix(df, "testing")
vaccine_matrix = generate_contact_matrix(df, "vacc")

In [ ]:
fig, ax = plot_histogram_distribution(mask_distribution, save_path="figures/Fig_3/masks_distribution.pdf")
fig, ax = plot_histogram_distribution(test_distribution, save_path="figures/Fig_3/testing_distribution.pdf")
fig, ax = plot_histogram_distribution(vaccine_distribution, save_path="figures/Fig_3/vaccine_distribution.pdf")

In [ ]:
fig, ax = plot_contact_matrix(mask_matrix, Lx, Ly, path="figures/Fig_3/mask_contact_matrix.pdf")
fig, ax = plot_contact_matrix(test_matrix, Lx, Ly, path="figures/Fig_3/testing_contact_matrix.pdf")
fig, ax = plot_contact_matrix(vaccine_matrix, Lx, Ly, path="figures/Fig_3/vaccine_contact_matrix.pdf")

In [ ]:
# calculate and print the range of the confidence interval:
R_M = bootstrap_mph(df, "masks", n_bootstrap=20000)
R_T = bootstrap_mph(df, "testing", n_bootstrap=20000)
R_V = bootstrap_mph(df, "vacc", n_bootstrap=20000)

print(f"[{np.round(R_M['M_CI'], 2)}, {np.round(R_M['P_CI'], 2)}, {np.round(R_M['H_CI'], 2)}]")
print(f"[{np.round(R_T['M_CI'], 2)}, {np.round(R_T['P_CI'], 2)}, {np.round(R_T['H_CI'], 2)}]")
print(f"[{np.round(R_V['M_CI'], 2)}, {np.round(R_V['P_CI'], 2)}, {np.round(R_V['H_CI'], 2)}]")

In [ ]:
homophilic_tendency = {"m": 0, "M": 6, "n": NB}
pol_range = {"m": 0, "M": 1, "n": NP}


betas = [0.15, 0.25, 0.4]
simulated_days = 3000


print("### SIR-M ###")
PS = 5
res_list_M = []
PARAMS["fixed_mean"] = mus["mean"][1]
for b in betas:
    PARAMS["beta_M"] = b
    M = sweep_two_parameters(
        model_module=SIRM,
        param1_name="beta_params",           # parameter 1 name
        param1_range=pol_range,    # parameter 1 range
        param2_name="homophilic_tendency",      # parameter 2 name
        param2_range=homophilic_tendency,         # parameter 2 range
        custom_base_params=PARAMS,
        simulated_days=simulated_days,
        population_size=PS,
        batch_size=2000
    )
    res_list_M.append(M)
    print(f"Completed beta_M = {b}")


print("### SIR-T ###")
res_list_T = []
PARAMS["fixed_mean"] = taus["mean"][1]
for b in betas:
    PARAMS["beta_M"] = b
    M = sweep_two_parameters(
        model_module=SIRT,
        param1_name="beta_params",           # parameter 1 name
        param1_range=pol_range,    # parameter 1 range
        param2_name="homophilic_tendency",      # parameter 2 name
        param2_range=homophilic_tendency,         # parameter 2 range
        custom_base_params=PARAMS,
        simulated_days=simulated_days,
        population_size=PS,
        batch_size=2000
    )
    res_list_T.append(M)
    print(f"Completed beta_M = {b}")

print("### SIR-V ###")
res_list_V = []
PARAMS["fixed_mean"] = xis["mean"][1]
for b in betas:
    PARAMS["beta_M"] = b
    M = sweep_two_parameters(
        model_module=SIRV,
        param1_name="beta_params",           # parameter 1 name
        param1_range=pol_range,    # parameter 1 range
        param2_name="homophilic_tendency",      # parameter 2 name
        param2_range=homophilic_tendency,         # parameter 2 range
        custom_base_params=PARAMS,
        simulated_days=simulated_days,
        population_size=PS,
        batch_size=2000
    )
    res_list_V.append(M)
    print(f"Completed beta_M = {b}")

In [ ]:
path_1_M_L = "figures/Fig_3/masks_L_08.pdf"
path_1_M_M = "figures/Fig_3/masks_M_08.pdf"
path_2_M_H = "figures/Fig_3/masks_H_08.pdf"


fig_R_M_1_L = plot_multiple_metrics(
    res_list_M[0], 
    metrics=["infections"],
    cmaps=cmaps, 
    contour_values=contour_values,
    contour_colors=contour_colors,
    final_params=final_params,
    rect_coords = rect_coords_M,
    save_path=path_1_M_L
)
fig_R_M_1_H = plot_multiple_metrics(
    res_list_M[1], 
    metrics=["infections"],
    cmaps=cmaps, 
    contour_values=contour_values,
    contour_colors=contour_colors,
    final_params=final_params,
    rect_coords = rect_coords_M,
    save_path=path_1_M_M
)
fig_R_M_2 = plot_multiple_metrics(
    res_list_M[2], 
    metrics=["infections"],
    cmaps=cmaps, 
    contour_values=contour_values,
    contour_colors=contour_colors,
    final_params=final_params,
    rect_coords = rect_coords_M,
    save_path=path_2_M_H
)

In [ ]:
path_1_T_M = "figures/Fig_3/tests_M.pdf"


fig_R_M_1_L = plot_multiple_metrics(
    res_list_T[0], 
    metrics=["infections"],
    cmaps=cmaps, 
    contour_values=contour_values,
    contour_colors=contour_colors,
    final_params=final_params,
    rect_coords = rect_coords_T,
    save_path=path_1_T_L
)
fig_R_M_1_H = plot_multiple_metrics(
    res_list_T[1], 
    metrics=["infections"],
    cmaps=cmaps, 
    contour_values=contour_values,
    contour_colors=contour_colors,
    final_params=final_params,
    rect_coords = rect_coords_T,
    save_path=path_1_T_M
)
fig_R_M_2 = plot_multiple_metrics(
    res_list_T[2], 
    metrics=["infections"],
    cmaps=cmaps, 
    contour_values=contour_values,
    contour_colors=contour_colors,
    final_params=final_params,
    rect_coords = rect_coords_T,
    save_path=path_1_T_H
)

In [ ]:
path_1_V_L = "figures/Fig_3/vaccines_L.pdf"
path_1_V_M = "figures/Fig_3/vaccines_M.pdf"
path_1_V_H = "figures/Fig_3/vaccines_H.pdf"


fig_R_M_1_L = plot_multiple_metrics(
    res_list_V[0], 
    metrics=["infections"],
    cmaps=cmaps, 
    contour_values=contour_values,
    contour_colors=contour_colors,
    final_params=final_params,
    rect_coords = rect_coords_V,
    save_path=path_1_V_L
)
fig_R_M_1_H = plot_multiple_metrics(
    res_list_V[1], 
    metrics=["infections"],
    cmaps=cmaps, 
    contour_values=contour_values,
    contour_colors=contour_colors,
    final_params=final_params,
    rect_coords = rect_coords_V,
    save_path=path_1_V_M
)
fig_R_M_2 = plot_multiple_metrics(
    res_list_V[2], 
    metrics=["infections"],
    cmaps=cmaps,
    contour_values=contour_values,
    contour_colors=contour_colors,
    final_params=final_params,
    rect_coords = rect_coords_V,
    save_path=path_1_V_H
)

In [ ]:
PARAMS["beta_M"] = 0.25
P_min_M, P_max_M = find_hpol_minmax(SIRM, mus, PARAMS)
P_min_T, P_max_T = find_hpol_minmax(SIRT, taus, PARAMS)
P_min_V, P_max_V = find_hpol_minmax(SIRV, xis, PARAMS)

In [ ]:
N_days = 2000
days = np.arange(0, N_days+1, 1)
colors_3 = ["#7570b3", "#d95f02", "#1b9e77"]

MINS_M, MAXS_M, BASES_M = calc_minmax_trajectories(SIRM, P_min_M, P_max_M, mus["mean"][1], PARAMS, simulated_days = N_days)
fig_comparison_M = plot_double_comparison(days, MINS_M, MAXS_M, BASES_M, "figures/Fig_3/comparison_M.pdf", Lx, Ly, color = colors_3[0])

avg_predicted = (MINS_M[0][-1] + MINS_M[1][-1] + MAXS_M[0][-1] + MAXS_M[1][-1]) / 2

avg_base = (BASES_M[0][-1] + BASES_M[1][-1])

perc_increase = (avg_predicted - avg_base)  * 100
print(f"increase in infections for T model: + {perc_increase:.2f} %")

In [ ]:
MINS_T, MAXS_T, BASES_T = calc_minmax_trajectories(SIRT, P_min_T, P_max_T, taus["mean"][1], PARAMS, simulated_days = N_days)
fig_comparison_T = plot_double_comparison(days, MINS_T, MAXS_T, BASES_T, "figures/Fig_3/comparison_T.pdf", Lx, Ly, color = colors_3[1])

avg_predicted = (MINS_T[0][-1] + MINS_T[1][-1] + MAXS_T[0][-1] + MAXS_T[1][-1]) / 2

avg_base = (BASES_T[0][-1] + BASES_T[1][-1])

perc_increase = (avg_predicted - avg_base) * 100
print(f"increase in infections for T model: + {perc_increase:.2f}%")

In [ ]:
MINS_V, MAXS_V, BASES_V = calc_minmax_trajectories(SIRV, P_min_V, P_max_V, xis["mean"][1], PARAMS, simulated_days = N_days)
fig_comparison_V = plot_double_comparison(days, MINS_V, MAXS_V, BASES_V, "figures/Fig_3/comparison_V.pdf", Lx, Ly, color = colors_3[2])

avg_predicted = (MINS_V[0][-1] + MINS_V[1][-1] + MAXS_V[0][-1] + MAXS_V[1][-1]) / 2

avg_base = (BASES_V[0][-1] + BASES_V[1][-1])

perc_increase = ((avg_predicted - avg_base)) * 100
print(f"Percentage increase in infections for V model: + {perc_increase:.2f}%")

# Things for Fig. 4:

In [ ]:

from copy import deepcopy

def run_one(R0, PARAMS, spec, model, pol_baseline, hom_baseline, simulated_days = 3000, sampling_points = 2):
    CB = deepcopy(PARAMS)
    
    pols = {"m": spec["pol"][0], "M": spec["pol"][2], "n": sampling_points}
    homs = {"m": spec["h"][0], "M": spec["h"][2], "n": sampling_points}


    CB["beta_M"] = R0 * CB["recovery_rate"]
    CB["fixed_mean"] = spec["mean"][1]
    # run the heterogeneous simulation
    RES_HET = sweep_two_parameters(
        model_module=model,
        param1_name="beta_params",              # parameter 1 name
        param1_range=pols,                      # parameter 1 range
        param2_name="homophilic_tendency",      # parameter 2 name
        param2_range=homs,                      # parameter 2 range
        custom_base_params=CB,
        simulated_days=simulated_days,
        population_size=5,
        batch_size= sampling_points*sampling_points
    )
    # run the homogeneous simulation
    RES_HOM = sweep_two_parameters(
        model_module=model,
        param1_name="beta_params",              # parameter 1 name
        param1_range=pol_baseline,              # parameter 1 range
        param2_name="homophilic_tendency",      # parameter 2 name
        param2_range=hom_baseline,              # parameter 2 range
        custom_base_params=CB,
        simulated_days=simulated_days,
        population_size=5,
        batch_size=1
    )
    HET = np.sum(RES_HET["final_state"]["R"], axis=2) + np.sum(RES_HET["final_state"]["I"], axis=2)
    HOM = np.sum(RES_HOM["final_state"]["R"], axis=2) + np.sum(RES_HOM["final_state"]["I"], axis=2)
    return HET, HOM

In [ ]:
R0s = np.linspace(1, 5, 30)
sirm_min_main = np.zeros(len(R0s))
sirm_max_main = np.zeros(len(R0s))
sirm_hom_main = np.zeros(len(R0s))

sirt_min_main = np.zeros(len(R0s))
sirt_max_main = np.zeros(len(R0s))
sirt_hom_main = np.zeros(len(R0s))

sirv_min_main = np.zeros(len(R0s))
sirv_max_main = np.zeros(len(R0s))
sirv_hom_main = np.zeros(len(R0s))
print(mus)
print(taus)
print(xis)
for i, R0 in enumerate(R0s):
    print(i, R0)
    het, hom = run_one(R0, PARAMS, mus, SIRM, [0.001], [0], simulated_days=1000, sampling_points=2)
    sirm_min_main[i] = np.min(np.array(het))
    sirm_max_main[i] = np.max(np.array(het))
    sirm_hom_main[i] = hom[0][0]

    het, hom = run_one(R0, PARAMS, taus, SIRT, [0.001], [0], simulated_days=1000, sampling_points=2)
    sirt_min_main[i] = np.min(np.array(het))
    sirt_max_main[i] = np.max(np.array(het))
    sirt_hom_main[i] = hom[0][0]

    het, hom = run_one(R0, PARAMS, xis, SIRV, [0.001], [0], simulated_days=1000, sampling_points=2)
    sirv_min_main[i] = np.min(np.array(het))
    sirv_max_main[i] = np.max(np.array(het))
    sirv_hom_main[i] = hom[0][0]


In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(3, 2))

M_min_main = (sirm_min_main - sirm_hom_main)*100
M_max_main = (sirm_max_main - sirm_hom_main)*100

T_min_main = (sirt_min_main - sirt_hom_main)*100
T_max_main = (sirt_max_main - sirt_hom_main)*100

V_min_main = (sirv_min_main - sirv_hom_main)*100
V_max_main = (sirv_max_main - sirv_hom_main)*100

LW = 0.5

colors_3 = ["#7570b3", "#d95f02", "#1b9e77"]

axs.fill_between(R0s, M_min_main, M_max_main, color=colors_3[0], alpha=1)
axs.plot(R0s, M_min_main, label="SIR-M min", color='black', linewidth=LW)
axs.plot(R0s, M_max_main, label="SIR-M max", color='black', linewidth=LW)


axs.fill_between(R0s, T_min_main, T_max_main, color=colors_3[1], alpha=1)
axs.plot(R0s, T_min_main, label="SIR-T min", color='black', linewidth=LW)
axs.plot(R0s, T_max_main, label="SIR-T max", color='black', linewidth=LW)



axs.axhline(0, color='black', linestyle='--', linewidth=1)
axs.fill_between(R0s, V_min_main, V_max_main, color=colors_3[2], alpha=1)
axs.plot(R0s, V_min_main, label="SIR-V min", color='black', linewidth=LW)
axs.plot(R0s, V_max_main, label="SIR-V max", color='black', linewidth=LW)



axs.axvline(2.5, color='black', linestyle=':', linewidth=1)

axs.set_ylim(-50, 50)
axs.set_xlim(1, 5)

axs.set_xticks([1, 3, 5])
axs.set_yticks([-50, 0, 50])

axs.set_xticklabels([])
axs.set_yticklabels([])

# remove top and right spines
axs.spines['top'].set_visible(False)
axs.spines['right'].set_visible(False)


#fig.savefig("figures/Fig_4/percentage_increase.pdf", dpi=300, bbox_inches='tight')

In [ ]:
# find the maximum percentage increase across all R0s for each model, at what value of R0 does it occur?
max_increase_M = np.max(M_max_main)
max_increase_T = np.max(T_max_main)
max_increase_V = np.max(V_max_main)

maxR0_M = R0s[np.argmax(M_max_main)]
maxR0_T = R0s[np.argmax(T_max_main)]
maxR0_V = R0s[np.argmax(V_max_main)]

# same but for the minimum percentage increase
min_increase_M = np.min(M_min_main)
min_increase_T = np.min(T_min_main)
min_increase_V = np.min(V_min_main)

minR0_M = R0s[np.argmin(M_min_main)]
minR0_T = R0s[np.argmin(T_min_main)]
minR0_V = R0s[np.argmin(V_min_main)]

print(f"Maximum percentage increase for SIR-M: {max_increase_M:.2f}% at R0 = {maxR0_M:.2f}")
print(f"Maximum percentage increase for SIR-T: {max_increase_T:.2f}% at R0 = {maxR0_T:.2f}")
print(f"Maximum percentage increase for SIR-V: {max_increase_V:.2f}% at R0 = {maxR0_V:.2f}")
print("_________")

print(f"Minimum percentage increase for SIR-M: {min_increase_M:.2f}% at R0 = {minR0_M:.2f}")
print(f"Minimum percentage increase for SIR-T: {min_increase_T:.2f}% at R0 = {minR0_T:.2f}")
print(f"Minimum percentage increase for SIR-V: {min_increase_V:.2f}% at R0 = {minR0_V:.2f}")

# Things for SI

In [ ]:
temp = read_json("./parameters.json")
mus, taus, xis, PARAMS = temp["mus"], temp["taus"], temp["xis"], temp["PARAMS"]

In [ ]:
NB = 100
NP = 100
homophilic_tendency = {"m": 0, "M": 6, "n": NB}
pol_range = {"m": 0, "M": 1, "n": NP}

print("### SIR-M ###")
PS = 25
res_list_M_25 = []
betas = np.linspace(0.15, 0.4, 25)
betas = [0.11, 0.15, 0.2, 
         0.25, 0.3, 0.35, 
         0.4, 0.45, 0.5]

for b in betas:
    PARAMS["beta_M"] = b
    M = sweep_two_parameters(
        model_module=SIRM,
        param1_name="beta_params",           # parameter 1 name
        param1_range=pol_range,    # parameter 1 range
        param2_name="homophilic_tendency",      # parameter 2 name
        param2_range=homophilic_tendency,         # parameter 2 range
        custom_base_params=PARAMS,
        simulated_days=1000,
        population_size=PS,
        batch_size=2000
    )
    res_list_M_25.append(M)
    print(f"Completed beta_M = {b}")

print("### SIR-T ###")
res_list_T_25 = []

for b in betas:
    PARAMS["beta_M"] = b
    M = sweep_two_parameters(
        model_module=SIRT,
        param1_name="beta_params",           # parameter 1 name
        param1_range=pol_range,    # parameter 1 range
        param2_name="homophilic_tendency",      # parameter 2 name
        param2_range=homophilic_tendency,         # parameter 2 range
        custom_base_params=PARAMS,
        simulated_days=1000,
        population_size=PS,
        batch_size=2000
    )
    res_list_T_25.append(M)
    print(f"Completed beta_M = {b}")

print("### SIR-V ###")
res_list_V_25 = []
for b in betas:
    PARAMS["beta_M"] = b
    M = sweep_two_parameters(
        model_module=SIRV,
        param1_name="beta_params",           # parameter 1 name
        param1_range=pol_range,    # parameter 1 range
        param2_name="homophilic_tendency",      # parameter 2 name
        param2_range=homophilic_tendency,         # parameter 2 range
        custom_base_params=PARAMS,
        simulated_days=1000,
        population_size=PS,
        batch_size=2000
    )
    res_list_V_25.append(M)
    print(f"Completed beta_M = {b}")

In [ ]:
def plot_9(res_list):
    fig, axes = plt.subplots(3,3, figsize=(7.0, 7.0), constrained_layout=True)
    for i, ax in enumerate(axes.flatten()):
        TI = np.sum(res_list[i]["final_state"]["I"] + res_list[i]["final_state"]["R"], axis=2)

        im = ax.imshow(
            TI, vmin = 0, vmax = 1, cmap = my_map,
            origin='lower', extent=[0, 1, 0, 20],
            aspect='auto')
        ax.set_title(f"{betas[i]:.3f}")
        
        # Find minimum polarization for each homophily level
        min_pol_curve = []
        homophily_values = []
        
        for h_idx in range(TI.shape[0]):  # iterate over homophily axis (rows)
            # Find minimum polarization index for this homophily level
            pol_idx = np.nanargmin(TI[h_idx, :])
            
            # Convert indices to actual values
            min_pol = pol_range["m"] + (pol_range["M"] - pol_range["m"]) * (pol_idx / (pol_range["n"] - 1))
            homophily = homophilic_tendency["m"] + (homophilic_tendency["M"] - homophilic_tendency["m"]) * (h_idx / (homophilic_tendency["n"] - 1))
            
            min_pol_curve.append(min_pol)
            homophily_values.append(homophily)
        
        # Plot the curve
        #ax.plot(min_pol_curve, homophily_values, color="green", linewidth=1)
        ax.set_xticks([0, 0.5, 1.0])
        ax.set_yticks([0, 10, 20])
    for i in range(3):
        axes[2,i].set_xlabel("Polarization")
    for i in range(3):
        axes[i,0].set_ylabel("Homophily")

    # set an overall title:
    return fig, axes

In [ ]:
f, a = plot_9(res_list_M_25)
f.savefig("figures/Fig_SI/many_beta_M.pdf")
f, a = plot_9(res_list_T_25)
f.savefig("figures/Fig_SI/many_beta_T.pdf")
f, a = plot_9(res_list_V_25)
f.savefig("figures/Fig_SI/many_beta_V.pdf")


In [ ]:
def find_critical_homophily(res, threshold=0):
    """
    For each beta_M result, find the max homophily where Jensen effect holds.
    
    res: result from sweep_two_parameters
    Returns: array of critical h values, shape (n_betas,)
    
    The grid is (n_pol x n_hom), R has shape (n_hom, n_pol, pop_size).
    Jensen works at a given h if dR/d_pol < threshold (increasing pol -> fewer cases).
    """
    R = np.array(res["final_state"]["R"])          # (n_hom, n_pol, pop_size)
    h_vals = np.array(res["parameter_grid"]["param2_vals"][:, 0])  # (n_hom,)
    
    total_R = R.sum(axis=-1)   # (n_hom, n_pol) - total recovered = total cases
    
    # Slope of R along polarization axis for each h
    slopes = np.diff(total_R, axis=1)  # (n_hom, n_pol-1)
    
    # Jensen works at h if ALL slopes are negative (monotonically decreasing)
    jensen_holds = np.all(slopes < threshold, axis=1)  # (n_hom,)
    
    # Find max h where Jensen holds
    valid_h = h_vals[jensen_holds]
    return valid_h.max() if len(valid_h) > 0 else 0.0

NB = 100
NP = 100
homophilic_tendency = {"m": 0, "M": 6, "n": NB}
pol_range = {"m": 0, "M": 1, "n": NP}

print("### SIR-M ###")
PS = 50
res_list_M_slim = []
betas = np.linspace(0.15, 0.3, 30)
for b in betas:
    PARAMS["beta_M"] = b
    M = sweep_two_parameters(
        model_module=SIRM,
        param1_name="beta_params",           # parameter 1 name
        param1_range=pol_range,    # parameter 1 range
        param2_name="homophilic_tendency",      # parameter 2 name
        param2_range=homophilic_tendency,         # parameter 2 range
        custom_base_params=PARAMS,
        simulated_days=500,
        population_size=PS,
        batch_size=2000
    )
    res_list_M_slim.append(M)
    print(f"Completed beta_M = {b}")



In [ ]:
critical_h_M = [find_critical_homophily(res) for res in res_list_M_slim]

fig, ax = plt.subplots(figsize=(7, 7/2.5))
ax.plot(betas*10, critical_h_M, 'o-')
ax.set_xlabel('$R_0^*$')
ax.set_ylabel('Critical homophily')
ax.set_title('Critical homophily vs $R_0^*$ for SIR-M')

ax.set_xticks([1.5, 2.0, 2.5, 3.0])
ax.set_yticks([0, 3, 6])
ax.set_yticklabels(['0', '3', '6+'])

ax.set_xlim(1.48, 3.02)
ax.set_ylim(-0.11, 6.11)

# remove top and right spines

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

#fig.tight_layout()
#fig.savefig("figures/Fig_SI/critical_homophily_SIR-M.pdf", dpi=300, bbox_inches='tight')

In [ ]:
NB = 50
NP = 50
homophilic_tendency = {"m": 0, "M": 20, "n": NB}
pol_range = {"m": 0, "M": 1, "n": NP}

print("### SIR-M ###")
PS = 20
res_list_M_25 = []
PARAMS["fixed_mean"] = mus["mean"][1]
betas = np.linspace(0.15, 0.4, 25)
betas = [0.11, 0.15, 0.2, 
         0.25, 0.3, 0.35, 
         0.4, 0.45, 0.5]
for b in betas:
    PARAMS["beta_M"] = b
    M = sweep_two_parameters(
        model_module=SIRM,
        param1_name="beta_params",           # parameter 1 name
        param1_range=pol_range,    # parameter 1 range
        param2_name="homophilic_tendency",      # parameter 2 name
        param2_range=homophilic_tendency,         # parameter 2 range
        custom_base_params=PARAMS,
        simulated_days=1000,
        population_size=PS,
        batch_size=2000
    )
    res_list_M_25.append(M)
    print(f"Completed beta_M = {b}")

print("### SIR-T ###")

PARAMS["fixed_mean"] = taus["mean"][1]
res_list_T_25 = []

for b in betas:
    PARAMS["beta_M"] = b
    M = sweep_two_parameters(
        model_module=SIRT,
        param1_name="beta_params",           # parameter 1 name
        param1_range=pol_range,    # parameter 1 range
        param2_name="homophilic_tendency",      # parameter 2 name
        param2_range=homophilic_tendency,         # parameter 2 range
        custom_base_params=PARAMS,
        simulated_days=1000,
        population_size=PS,
        batch_size=2000
    )
    res_list_T_25.append(M)
    print(f"Completed beta_M = {b}")

print("### SIR-V ###")

PARAMS["fixed_mean"] = xis["mean"][1]
res_list_V_25 = []
for b in betas:
    PARAMS["beta_M"] = b
    M = sweep_two_parameters(
        model_module=SIRV,
        param1_name="beta_params",           # parameter 1 name
        param1_range=pol_range,    # parameter 1 range
        param2_name="homophilic_tendency",      # parameter 2 name
        param2_range=homophilic_tendency,         # parameter 2 range
        custom_base_params=PARAMS,
        simulated_days=1000,
        population_size=PS,
        batch_size=2000
    )
    res_list_V_25.append(M)
    print(f"Completed beta_M = {b}")

PARAMS["fixed_mean"] = 0.5


In [ ]:
f, a = plot_9(res_list_M_25)
f, a = plot_9(res_list_T_25)
f, a = plot_9(res_list_V_25)


In [ ]:
fig, axes = plot_bootstrap_heatmap(R_M, R_T, R_V, bins = 150, figsize=(7.09, 7))
axes[(0,0)].set_title("Masks", fontsize=12)
axes[(0,1)].set_title("Testing", fontsize=12)
axes[(0,2)].set_title("Vaccines", fontsize=12)
for i in range(3):
    axes[(0,i)].set_xlabel("Mean", fontsize=10)
for i in range(2):
    axes[(i,0)].set_ylabel("Homophily", fontsize=10)
axes[(2,0)].set_ylabel("Mean", fontsize=10)
for i in range(3):
    axes[(2,i)].set_xlabel("Polarization", fontsize=10)


for i in range(3):
    axes[(0,i)].set_xlim(0.4, 0.85)
    axes[(1,i)].set_xlim(0.2, 0.65)
    axes[(2,i)].set_xlim(0.2, 0.65)

    axes[(0,i)].set_xticks([0.4, 0.6, 0.8])
    axes[(1,i)].set_xticks([0.2, 0.4,  0.6])
    axes[(2,i)].set_xticks([0.2, 0.4,  0.6])

for i in range(3):
    axes[(0,i)].set_ylim(1.25,3.75)
    axes[(1,i)].set_ylim(1.25,3.75)
    axes[(2,i)].set_ylim(0.4,0.85)
    axes[(2,i)].set_yticks([0.4, 0.6, 0.8])


fig.savefig("Figures/Fig_SI/Fig_SI_bootstrapping.pdf", bbox_inches='tight')

In [ ]:
R0s = np.linspace(1, 5, 100)
sirm_min = np.zeros((len(R0s), 3, 3, 3))
sirm_max = np.zeros((len(R0s),3,3, 3))
sirm_hom = np.zeros((len(R0s),3,3, 3))

sirt_min = np.zeros((len(R0s),3,3, 3))
sirt_max = np.zeros((len(R0s),3,3, 3))
sirt_hom = np.zeros((len(R0s),3,3, 3))

sirv_min = np.zeros((len(R0s),3,3, 3))
sirv_max = np.zeros((len(R0s),3,3, 3))
sirv_hom = np.zeros((len(R0s),3,3, 3))



pol_l = [0.2, 0.45, 0.7]
pol_h = [0.3, 0.55, 0.8]

hom_l = [0.5 , 2.5, 4.5]
hom_h = [1, 3, 5]


P = [[0.2, 0.25, 0.3],
      [1/3-0.05, 1/3, 1/3+0.05],
      [0.45, 0.5, 0.55]]


H = [[0.5, 0.75, 1],
      [2.5, 2.75, 3],
      [4.5, 4.75, 5]]

M = [[0.3, 0.3, 0.3],[0.5, 0.5,0.5],[0.7, 0.7,0.7]]


for k in range(3):
    for j in range(3):
        for m in range(3):
            pars = {"mean": M[m], "pol" : P[k], "h" : H[j]}
            for i, R0 in enumerate(R0s):
                  het, hom = run_one(R0, PARAMS, pars, SIRM, [0.001], [0], simulated_days=1000, sampling_points=2)
                  sirm_min[i][k][j][m] = np.min(np.array(het))
                  sirm_max[i][k][j][m] = np.max(np.array(het))
                  sirm_hom[i][k][j][m] = hom[0][0]

                  het, hom = run_one(R0, PARAMS, pars, SIRT, [0.001], [0], simulated_days=1000, sampling_points=2)
                  sirt_min[i][k][j][m] = np.min(np.array(het))
                  sirt_max[i][k][j][m] = np.max(np.array(het))
                  sirt_hom[i][k][j][m] = hom[0][0]

                  het, hom = run_one(R0, PARAMS, pars, SIRV, [0.001], [0], simulated_days=1000, sampling_points=2)
                  sirv_min[i][k][j][m] = np.min(np.array(het))
                  sirv_max[i][k][j][m] = np.max(np.array(het))
                  sirv_hom[i][k][j][m] = hom[0][0]

                  print(f"Completed R0 = {R0}, pol = {P[k]}, hom = {H[j]}, mean = {M[m]}")




In [ ]:
from src.utils.distributions import pol_mean_to_ab, my_beta_asymmetric

colors_m = ['#bdc9e1','#74a9cf','#0570b0']
colors_t = ['#d7b5d8','#df65b0','#ce1256']
colors_v = ['#b2e2e2','#66c2a4','#238b45']



Lxinches = 178 / 25.4
Lyinches = 178 / 25.4

fig = plt.figure(figsize=(Lxinches, Lyinches))
gs = fig.add_gridspec(2, 1, height_ratios=[0.15, 1], hspace=0.1)

gs_top = gs[0].subgridspec(1, 3, wspace=0.1)
gs_bot = gs[1].subgridspec(3, 3, hspace=0.3, wspace=0.3)

grays = ['#d9d9d9', '#969696', '#525252']

for k in range(3):
    gs_hist = gs_top[k].subgridspec(1, 3, wspace=0.05)
    for MM in range(3):
        ax_hist = fig.add_subplot(gs_hist[MM])
        a, b = pol_mean_to_ab(np.array(P[k][1]), np.array(M[MM][1]))
        n_bins = 5
        x = np.linspace(0, 1, n_bins)
        y = my_beta_asymmetric(a, b, n_bins)
        ax_hist.bar(x, y, width=1/n_bins*0.8, color=grays[MM], edgecolor='black', linewidth=0.3)
        ax_hist.set_xlim(-0.1, 1.1)
        ax_hist.set_xticks([])
        ax_hist.set_yticks([])
        if MM == 1:
            ax_hist.set_title(f"pol={P[k][1]:.2f}", fontsize=8)
        ax_hist.set_xlabel(f"m={M[MM][1]:.1f}", fontsize=8)

axs = [[fig.add_subplot(gs_bot[j, k]) for k in range(3)] for j in range(3)]

for MM in range(3):
    for k in range(3):
        for j in range(3):
            M_avg = ((sirm_min[:,k,j,MM] + sirm_max[:,k,j,MM]) / 2 - sirm_hom[:,k,j,MM]) * 100
            T_avg = ((sirt_min[:,k,j,MM] + sirt_max[:,k,j,MM]) / 2 - sirt_hom[:,k,j,MM]) * 100
            V_avg = ((sirv_min[:,k,j,MM] + sirv_max[:,k,j,MM]) / 2 - sirv_hom[:,k,j,MM]) * 100

            LW = 1.5
            if MM == 2:
                axs[j][k].plot(R0s, M_avg, color=colors_m[MM], linewidth=LW, linestyle='-', label = "SIR-M")
                axs[j][k].plot(R0s, T_avg, color=colors_t[MM], linewidth=LW, linestyle='--', label = "SIR-T")
                axs[j][k].plot(R0s, V_avg, color=colors_v[MM], linewidth=LW, linestyle=':', label = "SIR-V")
            else:
                axs[j][k].plot(R0s, M_avg, color=colors_m[MM], linewidth=LW, linestyle='-')
                axs[j][k].plot(R0s, T_avg, color=colors_t[MM], linewidth=LW, linestyle='--')
                axs[j][k].plot(R0s, V_avg, color=colors_v[MM], linewidth=LW, linestyle=':')
            axs[j][k].axhline(0, color='black', linestyle='--', linewidth=1)
            axs[j][k].axvline(2.5, color='black', linestyle=':', linewidth=1)
            axs[j][k].set_ylim(-50, 50)
            axs[j][k].set_xlim(1, 5)
            axs[j][k].set_xticks([1, 3, 5])
            axs[j][k].set_yticks([-50, 0, 50])

            if k == 0:
                axs[j][k].set_ylabel(f"hom={H[j][1]:.2f}", fontsize=10)
                axs[j][k].text(-0.25, 0.5, "Structural bias  $\\Delta I$", fontsize=8, transform=axs[j][k].transAxes, ha='center', va='center', rotation=90)
            if j == 2:
                axs[j][k].set_xlabel("$R_0^*$", fontsize=10)
            axs[j][k].legend(fontsize=6)

fig.canvas.draw()

for k in range(3):
    # Get the 3 histogram axes for this column
    hist_axes = [fig.axes[k*3 + MM] for MM in range(3)]
    
    # Get the corresponding main plot axis (first row, same column)
    main_ax = axs[0][k]
    
    # Get main axis position in figure coordinates
    main_pos = main_ax.get_position()
    
    # Get current hist group left and right (from first and last hist in group)
    left_pos = hist_axes[0].get_position()
    right_pos = hist_axes[2].get_position()
    
    total_width = right_pos.x1 - left_pos.x0
    single_width = total_width / 3
    
    for MM in range(3):
        new_x0 = main_pos.x0 + MM * (main_pos.width / 3)
        new_x1 = new_x0 + main_pos.width / 3
        old_pos = hist_axes[MM].get_position()
        hist_axes[MM].set_position([new_x0, old_pos.y0, main_pos.width / 3, old_pos.height])
plt.show()

fig.savefig("figures/Fig_SI/structural_bias_comparison.pdf", dpi=300, bbox_inches='tight')